# Error analysis and computational cost

Analyze the 50 saved subject-disjoint validation runs from the previous experiment. No training or new evaluation on the held-out test subjects is performed. Each validation trial appears twice, once per seed; its two predictions are related, not independent. Models and epochs were selected using validation results.

In [1]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, balanced_accuracy_score, f1_score

base=Path(r'D:\EEG Experiment Artifacts\2026-11-02')
source=base/'stability_5_models'
output=base/'evaluation';output.mkdir(parents=True,exist_ok=True)
protocol=json.loads((source/'protocol.json').read_text(encoding='utf8'))
models=['MSGM','REVE','EEGPT','LUNA','A00113']
assert set(models)==set(protocol['models']) and len(protocol['runs'])==10
records=[];confusions=[];sample_predictions=[]
for model in models:
    for split in protocol['runs']:
        fold,seed=split['fold'],split['seed']
        folder=source/'runs'/model/f'fold_{fold:02d}'/f'seed_{seed}'
        info=json.loads((folder/'metrics.json').read_text(encoding='utf8'))
        history=pd.read_csv(folder/'history.csv')
        with np.load(folder/'validation_predictions.npz') as data:
            logits=data['logits'];truth=data['labels'];subjects=data['subjects']
            predicted=(logits.mean(axis=1) if logits.ndim==3 else logits).argmax(axis=1)
            assert len(truth)==len(predicted)==len(subjects)
            assert set(subjects.tolist())==set(split['val'])
            assert info['split']==split and not set(split['train'])&set(split['val'])
            assert np.isclose(balanced_accuracy_score(truth,predicted),info['video_balanced_accuracy'])
            assert np.isclose(f1_score(truth,predicted,average='macro'),info['video_macro_f1'])
            cm=confusion_matrix(truth,predicted,labels=[0,1]);assert cm.sum()==len(truth)
            for actual in (0,1):
                confusions.append({'model':model,'fold':fold,'seed':seed,'actual':actual,'correct':int(cm[actual,actual]),'incorrect':int(cm[actual,1-actual]),'total':int(cm[actual].sum()),'recall':cm[actual,actual]/cm[actual].sum()})
            if 'trial_ids' in data.files:
                ids=data['trial_ids'];videos=np.r_[np.arange(1,13),np.arange(17,29)][ids%24]
                assert np.array_equal(ids//24,subjects)
            elif 'videos' in data.files:
                videos=data['videos']
            else:
                videos=None
            if videos is not None:
                sample_predictions.extend({'model':model,'fold':fold,'seed':seed,'subject':int(sub),'video':int(video),'actual':int(y),'predicted':int(pred)} for sub,video,y,pred in zip(subjects,videos,truth,predicted))
        records.append({'model':model,'fold':fold,'seed':seed,'accuracy_balanced':info['video_balanced_accuracy'],'macro_f1':info['video_macro_f1'],'best_epoch':info['best_epoch'],'epochs_run':len(history),'training_plus_validation_seconds':history['seconds'].sum(),'seconds_per_epoch':history['seconds'].mean()})
runs=pd.DataFrame(records);class_runs=pd.DataFrame(confusions);sample_predictions=pd.DataFrame(sample_predictions)
assert len(runs)==50 and len(class_runs)==100
assert all(runs.groupby('model').size()==10)
print('Verified:',len(runs),'runs across five folds and two seeds.')

Verified: 50 runs across five folds and two seeds.


## Classification errors

Class 0 is negative and class 1 is positive, according to the binary labels used in the saved experiments. Counts pool both seeds and five folds; the same video from one participant is represented twice. Recall is the mean across the ten runs.

In [2]:
class_table=class_runs.groupby(['model','actual']).agg(correct=('correct','sum'),incorrect=('incorrect','sum'),total=('total','sum'),mean_recall=('recall','mean'),recall_std=('recall','std')).reset_index()
class_table['class']=class_table['actual'].map({0:'negative',1:'positive'})
class_table=class_table[['model','class','correct','incorrect','total','mean_recall','recall_std']]
class_table.to_csv(output/'classification_errors.csv',index=False)
display(class_table.round(4))

,model,class,correct,incorrect,total,mean_recall,recall_std
0,A00113,negative,1410,1254,2664,0.5291,0.0996
1,A00113,positive,1610,1054,2664,0.6045,0.1080
2,EEGPT,negative,1446,1218,2664,0.5426,0.0715
3,EEGPT,positive,1660,1004,2664,0.6234,0.0780
4,LUNA,negative,1546,1118,2664,0.5799,0.0496
5,LUNA,positive,1558,1106,2664,0.5851,0.0432
6,MSGM,negative,1574,1090,2664,0.5903,0.1043
7,MSGM,positive,1652,1012,2664,0.6207,0.0986
8,REVE,negative,1524,1140,2664,0.5711,0.0765
9,REVE,positive,1609,1055,2664,0.6050,0.0707


## Shared errors

Saved trial identifiers are available for REVE, EEGPT, LUNA and A00113. The MSGM file does not contain video IDs, so it is excluded from sample-level matching. Common errors below require the same fold, seed, subject and video, not merely the same class.

In [3]:
key=['fold','seed','subject','video']
assert not sample_predictions.duplicated(key+['model']).any()
assert sample_predictions.groupby(key)['actual'].nunique().eq(1).all()
sample_predictions['error']=(sample_predictions['actual']!=sample_predictions['predicted'])
shared=sample_predictions.groupby(key).agg(actual=('actual','first'),models_evaluated=('model','nunique'),models_wrong=('error','sum')).reset_index()
assert shared['models_evaluated'].eq(4).all()
shared=shared[shared['models_wrong']==4].sort_values(key)
shared.to_csv(output/'shared_errors_four_models.csv',index=False)
print('Shared errors among the four models:',len(shared),'of',len(sample_predictions)//4,'seed-specific predictions.')
display(shared.head(10))

Shared errors among the four models: 774 of 5328 seed-specific predictions.


,fold,seed,subject,video,actual,models_evaluated,models_wrong
13,0,42,12,18,1,4,4
16,0,42,12,21,1,4,4
17,0,42,12,22,1,4,4
19,0,42,12,24,1,4,4
48,0,42,22,1,0,4,4
49,0,42,22,2,0,4,4
50,0,42,22,3,0,4,4
51,0,42,22,4,0,4,4
55,0,42,22,8,0,4,4
58,0,42,22,11,0,4,4


## Computational cost

Each recorded epoch includes training and validation. For the three frozen encoders, the timing covers only the linear classifier and uses precomputed embeddings: encoder inference, preprocessing and feature extraction are excluded. Therefore these times are not end-to-end comparisons. Parameter counts for these three models describe the trainable linear classifier only, not the full pretrained encoder.

In [4]:
import torch
sys.path.insert(0,str(base/'mamba_sources'/'MSGM'))
from msgm.model import MSGM, REGION_MAP_32_10
region_ids=list(REGION_MAP_32_10[:16])+list(REGION_MAP_32_10[18:])
msgm=MSGM(num_channels=30,num_features=5,num_classes=2,scale_lengths=(30,),region_ids=region_ids)
msgm_params=sum(p.numel() for p in msgm.parameters() if p.requires_grad)
checkpoint=torch.load(source/'runs'/'MSGM'/'fold_00'/'seed_42'/'best_model.pt',map_location='cpu',weights_only=True)
assert set(msgm.state_dict())==set(checkpoint['state_dict'])
graph_metrics=json.loads((base/'graph_experiments'/'runs'/'A00113'/'metrics.json').read_text(encoding='utf8'))
parameters={'MSGM':msgm_params,'A00113':graph_metrics['parameters']}
for model in ['REVE','EEGPT','LUNA']:
    state=torch.load(source/'runs'/model/'fold_00'/'seed_42'/'best_model.pt',map_location='cpu',weights_only=True)['state_dict']
    assert set(state)=={'weight','bias'}
    parameters[model]=sum(v.numel() for v in state.values())
cost=runs.groupby('model').agg(mean_epochs=('epochs_run','mean'),mean_seconds_per_epoch=('seconds_per_epoch','mean'),mean_seconds_per_run=('training_plus_validation_seconds','mean')).reset_index()
cost['trainable_parameters']=cost['model'].map(parameters)
cost['parameter_scope']=np.where(cost['model'].isin(['REVE','EEGPT','LUNA']),'linear head only','model')
cost.to_csv(output/'computational_cost.csv',index=False)
display(cost.round(2))

,model,mean_epochs,mean_seconds_per_epoch,mean_seconds_per_run,trainable_parameters,parameter_scope
0,A00113,10.7,3.01,32.18,24850,model
1,EEGPT,14.6,0.05,0.73,1026,linear head only
2,LUNA,12.6,0.04,0.46,514,linear head only
3,MSGM,14.0,20.97,292.04,154362,model
4,REVE,13.2,0.02,0.27,1026,linear head only


## Comparison for interpretation

The table repeats the earlier validation comparison for context; this cell does not run another experiment or statistical test. Values are averages across the saved ten paired fold/seed runs. Parameter counts and timings must be read with the caveats above.

In [5]:
comparison=runs.groupby('model').agg(mean_balanced_accuracy=('accuracy_balanced','mean'),std_balanced_accuracy=('accuracy_balanced','std'),mean_macro_f1=('macro_f1','mean'),std_macro_f1=('macro_f1','std')).reset_index()
comparison=comparison.merge(cost,on='model').sort_values('mean_balanced_accuracy',ascending=False)
comparison.to_csv(output/'comparison.csv',index=False)
display(comparison.round(4))

,model,mean_balanced_accuracy,std_balanced_accuracy,mean_macro_f1,std_macro_f1,mean_epochs,mean_seconds_per_epoch,mean_seconds_per_run,trainable_parameters,parameter_scope
3,MSGM,0.6055,0.0193,0.6018,0.0217,14.0,20.9712,292.0366,154362,model
4,REVE,0.5881,0.0120,0.5859,0.0138,13.2,0.0205,0.2656,1026,linear head only
1,EEGPT,0.5830,0.0237,0.5803,0.0251,14.6,0.0509,0.7326,1026,linear head only
2,LUNA,0.5825,0.0140,0.5817,0.0144,12.6,0.0367,0.4613,514,linear head only
0,A00113,0.5668,0.0117,0.5620,0.0110,10.7,3.0081,32.1759,24850,model
